# Step 0 — Dataset Preparation ✅

| | |
|---|---|
| **Input** | nuScenes v1.0-mini dataset — set `DATA_ROOT` in `config.py` |
| **Outputs** | `output/step_0/sample_XXXX/sensor_meta.json` — all 12 sensor channels per sample |
| | `output/step_0/sample_XXXX/ego_pose.json` — ego vehicle pose |
| | `output/step_0/sample_XXXX/timestamp.txt` — capture timestamp |
| | `output/step_0/samples_index.json` — **master index used by all downstream steps** |
| | `output/step_0/samples_report.csv` — summary CSV for inspection |
| **Used by** | Steps 1.1, 1.2, 1.3, 2.1, 2.2, 2.3, 4.4, 5 |

---

### Before running
1. Make sure `config.py` is in the **same folder as this notebook**
2. Open `config.py` and set `DATA_ROOT` to your nuScenes dataset path
3. Run cells top to bottom

In [1]:
import os
print(os.getcwd())

F:\Sensor fusion Research


In [2]:
from config import DATA_ROOT
print(list(DATA_ROOT.iterdir()))


config.py loaded. PROJECT_ROOT = F:\Sensor fusion Research
DATA_ROOT   = F:\Sensor fusion Research\DATA SET\archive
OUTPUT_ROOT = F:\Sensor fusion Research\output
[WindowsPath('F:/Sensor fusion Research/DATA SET/archive/.v1.0-mini.txt'), WindowsPath('F:/Sensor fusion Research/DATA SET/archive/maps'), WindowsPath('F:/Sensor fusion Research/DATA SET/archive/samples'), WindowsPath('F:/Sensor fusion Research/DATA SET/archive/sweeps'), WindowsPath('F:/Sensor fusion Research/DATA SET/archive/v1.0-mini')]


In [3]:
import os

print(os.path.exists("output"))

True


In [4]:
import os

print(os.listdir("output"))

['step_0', 'step_1', 'step_2', 'step_3', 'step_4', 'step_5', 'step_6']


In [5]:
import os

print(os.listdir("output/step_0"))

[]


In [6]:
import sys
print(sys.executable)

D:\anaconda\python.exe


In [7]:
import sys

print("Python:", sys.executable)
print()

try:
    import nuscenes
    print("SUCCESS!")
    print("nuscenes imported from:")
    print(nuscenes.__file__)
except Exception as e:
    print("FAILED:")
    print(e)

Python: D:\anaconda\python.exe

SUCCESS!
nuscenes imported from:
C:\Users\DS\AppData\Roaming\Python\Python312\site-packages\nuscenes\__init__.py


In [8]:
import site

print(site.getsitepackages())
print()
print(site.getusersitepackages())

['D:\\anaconda', 'D:\\anaconda\\Lib\\site-packages']

C:\Users\DS\AppData\Roaming\Python\Python312\site-packages


In [9]:
# ─────────────────────────────────────────────────────────────────
# CELL 1 — Verify config.py exists and DATA_ROOT is set correctly
# config.py should already be in the same folder as this notebook.
# If it is missing, this cell tells you exactly what to do.
# ─────────────────────────────────────────────────────────────────

import os
from pathlib import Path

if not Path("config.py").exists():
    raise FileNotFoundError(
        "config.py not found in this folder.\n"
        "Please download config.py from the repo root and place it "
        "in the same folder as this notebook, then set DATA_ROOT inside it."
    )

# Import and validate
from config import DATA_ROOT, NUSCENES_VERSION, STEP0_DIR

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"DATA_ROOT does not exist: {DATA_ROOT}\n"
        "Open config.py and set DATA_ROOT to your nuScenes dataset folder."
    )

print(f"✅ config.py found")
print(f"✅ DATA_ROOT  : {DATA_ROOT}")
print(f"✅ STEP0_DIR  : {STEP0_DIR}")
print(f"✅ Version    : {NUSCENES_VERSION}")

✅ config.py found
✅ DATA_ROOT  : F:\Sensor fusion Research\DATA SET\archive
✅ STEP0_DIR  : F:\Sensor fusion Research\output\step_0
✅ Version    : v1.0-mini


In [10]:
# ─────────────────────────────────────────────────────────────────
# CELL 2 — Main pipeline
# Loads nuScenes, extracts metadata for all 12 sensors per sample,
# saves sensor_meta.json / ego_pose.json / timestamp.txt per sample,
# and writes the master samples_index.json used by all downstream steps.
# ─────────────────────────────────────────────────────────────────

import json
from pathlib import Path
from nuscenes.nuscenes import NuScenes
from config import DATA_ROOT, NUSCENES_VERSION, STEP0_DIR

output_dir = STEP0_DIR
output_dir.mkdir(parents=True, exist_ok=True)

# ── Initialise nuScenes ────────────────────────────────────────────
nusc = NuScenes(version=NUSCENES_VERSION, dataroot=str(DATA_ROOT), verbose=True)
print("Number of samples:", len(nusc.sample))
print("DATA_ROOT:", DATA_ROOT)
samples_index = {}

# ── Iterate every sample ───────────────────────────────────────────
for idx, sample in enumerate(nusc.sample):
    sample_token  = sample['token']
    sample_id     = f'sample_{idx:04d}'
    sample_folder = output_dir / sample_id
    sample_folder.mkdir(exist_ok=True)

    sensor_meta = {
        "sample_token": sample_token,
        "sensor_channels": {}
    }

    # Extract metadata for all 12 sensors (1 LiDAR + 5 Radar + 6 Camera)
    for sensor_channel, data_token in sample['data'].items():
        sd = nusc.get('sample_data', data_token)
        cs = nusc.get('calibrated_sensor', sd['calibrated_sensor_token'])
        ep = nusc.get('ego_pose', sd['ego_pose_token'])

        sensor_meta["sensor_channels"][sensor_channel] = {
            "filename"                  : sd['filename'],
            "timestamp"                 : sd['timestamp'],
            "modality"                  : sd['sensor_modality'],
            "calibrated_sensor_token"   : sd['calibrated_sensor_token'],
            "ego_pose_token"            : sd['ego_pose_token'],
            "sensor_to_ego_translation" : cs['translation'],
            "sensor_to_ego_rotation"    : cs['rotation'],
            "ego_pose": {
                "translation": ep['translation'],
                "rotation"   : ep['rotation']
            }
        }

    # Timestamp from first available sensor
    timestamp = list(sensor_meta["sensor_channels"].values())[0]['timestamp']

    # Save timestamp.txt
    with open(sample_folder / 'timestamp.txt', 'w') as f:
        f.write(str(timestamp))

    # Save ego_pose.json — prefer LIDAR_TOP, fallback to first sensor
    ego_pose = (
        sensor_meta["sensor_channels"]["LIDAR_TOP"]["ego_pose"]
        if "LIDAR_TOP" in sensor_meta["sensor_channels"]
        else list(sensor_meta["sensor_channels"].values())[0]["ego_pose"]
    )
    with open(sample_folder / 'ego_pose.json', 'w') as f:
        json.dump(ego_pose, f, indent=2)

    # Save sensor_meta.json
    with open(sample_folder / 'sensor_meta.json', 'w') as f:
        json.dump(sensor_meta, f, indent=2)

    # Master index — store RELATIVE folder name only (NOT absolute path)
    # Downstream notebooks reconstruct full path as: STEP0_DIR / info["folder"]
    samples_index[sample_id] = {
        "sample_token" : sample_token,
        "timestamp"    : timestamp,
        "folder"       : sample_id,
        "sensors"      : list(sensor_meta["sensor_channels"].keys())
    }

# ── Save master index ──────────────────────────────────────────────
index_path = output_dir / 'samples_index.json'
with open(index_path, 'w') as f:
    json.dump(samples_index, f, indent=2)

# ── Validate ───────────────────────────────────────────────────────
assert len(samples_index) == len(nusc.sample), \
    f"Expected {len(nusc.sample)} samples, got {len(samples_index)}"

print(f"\n✅ Step 0 complete: {len(samples_index)} samples organised.")
print(f"📄 Index saved to : {index_path}")

Loading NuScenes tables for version v1.0-mini...
23 category,
8 attribute,
4 visibility,
911 instance,
12 sensor,
120 calibrated_sensor,
31206 ego_pose,
8 log,
10 scene,
404 sample,
31206 sample_data,
18538 sample_annotation,
4 map,
Done loading in 0.338 seconds.
Reverse indexing ...
Done reverse indexing in 0.1 seconds.
Number of samples: 404
DATA_ROOT: F:\Sensor fusion Research\DATA SET\archive

✅ Step 0 complete: 404 samples organised.
📄 Index saved to : F:\Sensor fusion Research\output\step_0\samples_index.json


In [11]:
# ─────────────────────────────────────────────────────────────────
# CELL 3 — Load and preview the master index
# ─────────────────────────────────────────────────────────────────

import json
from config import STEP0_DIR

with open(STEP0_DIR / 'samples_index.json') as f:
    index = json.load(f)

print(f"✅ Loaded {len(index)} samples from index.")
print("\nSample preview (sample_0003):")
print(json.dumps(index['sample_0003'], indent=2))

✅ Loaded 404 samples from index.

Sample preview (sample_0003):
{
  "sample_token": "e0845f5322254dafadbbed75aaa07969",
  "timestamp": 1532402929175111,
  "folder": "sample_0003",
  "sensors": [
    "RADAR_FRONT",
    "RADAR_FRONT_LEFT",
    "RADAR_FRONT_RIGHT",
    "RADAR_BACK_LEFT",
    "RADAR_BACK_RIGHT",
    "LIDAR_TOP",
    "CAM_FRONT",
    "CAM_FRONT_RIGHT",
    "CAM_BACK_RIGHT",
    "CAM_BACK",
    "CAM_BACK_LEFT",
    "CAM_FRONT_LEFT"
  ]
}


In [12]:
# ─────────────────────────────────────────────────────────────────
# CELL 4 — Sensor access verification
# Checks LiDAR (1), Radar (5), Camera (6) channels for a sample
# ─────────────────────────────────────────────────────────────────

import json
from config import STEP0_DIR

sample_id     = "sample_0003"    # ← change to inspect any sample
sample_folder = STEP0_DIR / sample_id

with open(sample_folder / "sensor_meta.json") as f:
    sensor_meta = json.load(f)

ch = sensor_meta["sensor_channels"]

# LiDAR
if "LIDAR_TOP" in ch:
    print(f"✅ LiDAR  → {ch['LIDAR_TOP']['filename']}")
else:
    print(f"❌ LiDAR not found")

# Radar (expect 5)
radar_ch = {k: v for k, v in ch.items() if "RADAR" in k}
print(f"\n✅ Radar  → {len(radar_ch)}/5 channels:")
for k, v in radar_ch.items():
    print(f"   {k:20s} → {v['filename']}")

# Camera (expect 6)
cam_ch = {k: v for k, v in ch.items() if "CAM_" in k}
print(f"\n✅ Camera → {len(cam_ch)}/6 channels:")
for k, v in cam_ch.items():
    print(f"   {k:25s} → {v['filename']}")

# Assert all 12 present
assert len(ch) == 12, f"Expected 12 sensor channels, got {len(ch)}"
print(f"\n✅ All 12 sensor channels confirmed for {sample_id}.")

✅ LiDAR  → samples/LIDAR_TOP/n015-2018-07-24-11-22-45+0800__LIDAR_TOP__1532402929197353.pcd.bin

✅ Radar  → 5/5 channels:
   RADAR_FRONT          → samples/RADAR_FRONT/n015-2018-07-24-11-22-45+0800__RADAR_FRONT__1532402929175111.pcd
   RADAR_FRONT_LEFT     → samples/RADAR_FRONT_LEFT/n015-2018-07-24-11-22-45+0800__RADAR_FRONT_LEFT__1532402929192815.pcd
   RADAR_FRONT_RIGHT    → samples/RADAR_FRONT_RIGHT/n015-2018-07-24-11-22-45+0800__RADAR_FRONT_RIGHT__1532402929174655.pcd
   RADAR_BACK_LEFT      → samples/RADAR_BACK_LEFT/n015-2018-07-24-11-22-45+0800__RADAR_BACK_LEFT__1532402929198900.pcd
   RADAR_BACK_RIGHT     → samples/RADAR_BACK_RIGHT/n015-2018-07-24-11-22-45+0800__RADAR_BACK_RIGHT__1532402929171003.pcd

✅ Camera → 6/6 channels:
   CAM_FRONT                 → samples/CAM_FRONT/n015-2018-07-24-11-22-45+0800__CAM_FRONT__1532402929162460.jpg
   CAM_FRONT_RIGHT           → samples/CAM_FRONT_RIGHT/n015-2018-07-24-11-22-45+0800__CAM_FRONT_RIGHT__1532402929170339.jpg
   CAM_BACK_RIGHT    

In [13]:
# ─────────────────────────────────────────────────────────────────
# CELL 5 — CSV report: one row per sample
# ─────────────────────────────────────────────────────────────────

import json
import pandas as pd
from config import STEP0_DIR

report_path = STEP0_DIR / "samples_report.csv"

with open(STEP0_DIR / "samples_index.json") as f:
    index = json.load(f)

rows = []
for sample_id, info in sorted(index.items()):
    sample_folder = STEP0_DIR / info["folder"]   # relative → absolute via config
    with open(sample_folder / "sensor_meta.json") as f:
        meta = json.load(f)
    ch = meta["sensor_channels"]

    lidar_files  = {k: v["filename"] for k, v in ch.items() if "LIDAR" in k}
    radar_files  = {k: v["filename"] for k, v in ch.items() if "RADAR" in k}
    camera_files = {k: v["filename"] for k, v in ch.items() if "CAM_"  in k}

    rows.append({
        "sample_id"    : sample_id,
        "sample_token" : info["sample_token"],
        "timestamp"    : info["timestamp"],
        "num_lidar"    : len(lidar_files),
        "num_radar"    : len(radar_files),
        "num_camera"   : len(camera_files),
        "lidar_file"   : lidar_files.get("LIDAR_TOP", ""),
        "radar_front"  : radar_files.get("RADAR_FRONT", ""),
        "cam_front"    : camera_files.get("CAM_FRONT", ""),
    })

df = pd.DataFrame(rows)
df.to_csv(report_path, index=False)

assert len(df) == len(index), f"Row count mismatch: {len(df)} vs {len(index)}"
print(f"✅ CSV report saved: {report_path}")
print(f"   {len(df)} rows | columns: {df.columns.tolist()}")
display(df.head())

✅ CSV report saved: F:\Sensor fusion Research\output\step_0\samples_report.csv
   404 rows | columns: ['sample_id', 'sample_token', 'timestamp', 'num_lidar', 'num_radar', 'num_camera', 'lidar_file', 'radar_front', 'cam_front']


,sample_id,sample_token,timestamp,num_lidar,num_radar,num_camera,lidar_file,radar_front,cam_front
0,sample_0000,ca9a282c9e77460f8360f564131a8af5,1532402927664178,1,5,6,samples/LIDAR_TOP/n015-2018-07-24-11-22-45+080...,samples/RADAR_FRONT/n015-2018-07-24-11-22-45+0...,samples/CAM_FRONT/n015-2018-07-24-11-22-45+080...
1,sample_0001,39586f9d59004284a7114a68825e8eec,1532402928114656,1,5,6,samples/LIDAR_TOP/n015-2018-07-24-11-22-45+080...,samples/RADAR_FRONT/n015-2018-07-24-11-22-45+0...,samples/CAM_FRONT/n015-2018-07-24-11-22-45+080...
2,sample_0002,356d81f38dd9473ba590f39e266f54e5,1532402928719956,1,5,6,samples/LIDAR_TOP/n015-2018-07-24-11-22-45+080...,samples/RADAR_FRONT/n015-2018-07-24-11-22-45+0...,samples/CAM_FRONT/n015-2018-07-24-11-22-45+080...
3,sample_0003,e0845f5322254dafadbbed75aaa07969,1532402929175111,1,5,6,samples/LIDAR_TOP/n015-2018-07-24-11-22-45+080...,samples/RADAR_FRONT/n015-2018-07-24-11-22-45+0...,samples/CAM_FRONT/n015-2018-07-24-11-22-45+080...
4,sample_0004,c923fe08b2ff4e27975d2bf30934383b,1532402929699442,1,5,6,samples/LIDAR_TOP/n015-2018-07-24-11-22-45+080...,samples/RADAR_FRONT/n015-2018-07-24-11-22-45+0...,samples/CAM_FRONT/n015-2018-07-24-11-22-45+080...
